[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhiksark/pycon-italy-2026-workshop/blob/solutions/solutions/03-reduction-triton.solution.ipynb)

# 03 · Reduction in Triton - _solution_

Reference solutions for the core (single-block sum) and bonus (multi-block sum) TODOs.

## Environment bootstrap

In [ ]:
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try: importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
ensure('triton')
import torch, triton
import triton.language as tl
assert torch.cuda.is_available(), 'No GPU detected.'
print(f'triton {triton.__version__}')

# Part 1 · Core: single-block sum

We start with a tensor short enough to fit in one block (BLOCK_SIZE = 1024). One program instance sums everything. `tl.sum(x, axis=0)` does the tree reduction across the block.

Two TODOs. Read the comments.

Mental picture: `tl.sum` collapses the block via a hidden binary tree.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb03-reduction-tree.png)

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/mask-load-1d.png)

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/tile-1d.png)

In [2]:
@triton.jit
def single_block_sum(x_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    total = tl.sum(x, axis=0)
    tl.store(out_ptr, total)


In [3]:
n = 1000
x = torch.rand(n, device='cuda', dtype=torch.float32)
out = torch.empty(1, device='cuda', dtype=torch.float32)
single_block_sum[(1,)](x, out, n, BLOCK_SIZE=1024)
torch.testing.assert_close(out[0], x.sum(), rtol=1e-3, atol=1e-3)
print(f'core ok | sum = {out.item():.4f}')

core ok | sum = 502.8703


# Part 2 · Challenge: multi-block sum

Tensors bigger than BLOCK_SIZE need more than one program. Each program sums its own block into a per-block partial. Then we sum the partials on the host (a tiny CPU reduction) - or with a second kernel call (left as homework).

Two TODOs.

In [4]:
@triton.jit
def multi_block_sum(x_ptr, partials_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    block_sum = tl.sum(x, axis=0)
    tl.store(partials_ptr + pid, block_sum)


In [5]:
n_big = 1 << 20
x_big = torch.rand(n_big, device='cuda', dtype=torch.float32)
BLOCK = 1024
grid = (triton.cdiv(n_big, BLOCK),)
partials = torch.empty(grid[0], device='cuda', dtype=torch.float32)
multi_block_sum[grid](x_big, partials, n_big, BLOCK_SIZE=BLOCK)
result = partials.sum()  # host-side final reduction (one transfer, one sum)
torch.testing.assert_close(result, x_big.sum(), rtol=1e-2, atol=1e-2)
print(f'bonus ok | sum = {result.item():.2f}')

bonus ok | sum = 524620.00


In [9]:
# --- benchmark: how close to peak HBM are we? ---
import sys, pathlib
# Make `from notebooks.shared.*` resolve whether Jupyter was launched
# from the repo root, from notebooks/, or anywhere below.
for _root in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_root / 'notebooks' / 'shared' / 'benchmark_utils.py').is_file():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
from notebooks.shared.benchmark_utils import median_seconds

def run_reduction():
    multi_block_sum[grid](x_big, partials, n_big, BLOCK_SIZE=BLOCK)
    partials.sum()
    torch.cuda.synchronize()

ms = median_seconds(run_reduction, runs=50, warmup=5) * 1e3
n_bytes = x_big.numel() * x_big.element_size()  # one streaming read of the input dominates
gbs = n_bytes / (ms * 1e-3) / 1e9
T4_PEAK_HBM_GBS = 320.0       # T4 datasheet
print(f'reduction: {ms:.3f} ms | {gbs:.1f} GB/s ({gbs / T4_PEAK_HBM_GBS * 100:.0f}% of T4 peak HBM)')


reduction: 0.077 ms | 54.2 GB/s (17% of T4 peak HBM)


## Teaching beat

Reductions on a GPU are tree-shaped, not linear. `tl.sum` hides the tree from you, but it is there - every warp does a local reduction in registers, then warps cooperate via shared memory.

Next: 2D, with halos.